# Initialization

In [ ]:
import copy
import csv
import json
from pathlib import Path
import pickle
import re
import sys
import time
import warnings

import openai
import pandas as pd
import torch
from openai import LengthFinishReasonError, OpenAI
from pydantic import BaseModel, field_validator
from torch.utils.data import Dataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load local classifier artifacts.
device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(
    "model",
    model_max_length=512,
    padding="max_length",
    truncation=True,
    local_files_only=True,
    use_fast=True,
 )
model = AutoModelForSequenceClassification.from_pretrained("model")

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    top_k=None,
    truncation=True,
    padding="max_length",
    max_length=512,
 )


def forward(text_input):
    """Run a forward pass and return logits and probabilities."""
    model.eval()
    with torch.no_grad():
        tokenized_input = tokenizer(
            text_input,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            max_length=512,
        )
        input_ids = tokenized_input["input_ids"].to(device)
        attention_mask = tokenized_input["attention_mask"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu()
        probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
        return logits.numpy(), probs


def mask_words_by_index(text, indexes):
    """Replace words at provided positions with [MASK]."""
    masked_words = []
    words = text.split()
    for i in indexes:
        if 0 <= i < len(words):
            masked_words.append(words[i])
            words[i] = "[MASK]"
    return " ".join(words), masked_words


def mask_words_by_words(text, words_to_mask):
    """Mask words by normalized lexical match (lowercased, punctuation removed)."""
    masked_words = []
    words = text.split()
    for i in range(len(words)):
        word1 = re.sub(r"[^\w\s]", "", words[i]).lower().strip()
        for wtm in words_to_mask:
            word2 = re.sub(r"[^\w\s]", "", wtm).lower().strip()
            if word1 == word2:
                masked_words.append(words[i])
                words[i] = "[MASK]"
    return " ".join(words), masked_words


def mask_and_predict(list_of_tokens, list_of_lists):
    """Evaluate model outputs for multiple token-index masking configurations."""
    masked_texts = []
    masked_tokens = []
    raw_outputs = []

    for li in list_of_lists:
        tokens = [str(t) for t in copy.deepcopy(list_of_tokens)]
        mt = []

        for i in li:
            if i < 0 or i >= len(tokens):
                print(f"Index {i} skipped.")
                continue
            mt.append(tokens[i])
            tokens[i] = "[MASK]"

        masked_text = "".join(tokens)
        masked_tokens.append(mt)
        masked_texts.append(masked_text)
        raw_outputs.append(forward(masked_text))

    outputs = [
        [float(ro[0][0][1]), float(ro[0][0][0]), float(ro[1][0][1])]
        for ro in raw_outputs
    ]
    return outputs, masked_tokens, masked_texts


def count_words(string):
    """Count whitespace-delimited tokens in a string."""
    return len(string.split())


# df = pd.read_csv("combined_result_cleaned.csv")

In [ ]:
df = pd.read_csv("feature_attributions.csv")

In [ ]:
for id in df["id"].unique():
    df_temp = df[df['id'] == id]
    print(mask_and_predict(list(df_temp['sv_token']), [[7, 8], [8]]))
    break

# GPT Explain

In [ ]:
with Path("info.json").open('r') as file:
    data = json.load(file)

GPT_MODEL = "gpt-4o-2024-11-20"

TEMPERATURE = 0

PRESENCE_PENALTY = 0
FREQUENCY_PENALTY = 0

NUM_LIST_MIN = 10
NUM_LIST_MAX = 30
NUM_ITEM = 5

MIN_I = 5
MAX_I = 15
FIXED_I = 10

client = OpenAI(
    api_key=data['api_key'],
    organization=data['org_id'],
    project=data['project_id']
)

PROMPT_ROLE = f"""
You are a machine learning model explainer.
"""
PROMPT_TASK = f"""
You are tasked with explaining binary text classification encoder-only transformer model's predictions by perturbing its input tokens, similar to perturbation based XAI frameworks like LIME or SHAP.
"""

PROMPT_CRITERIA = f"""
Manual appraisal criteria:
All of these criteria must be met to be rated as being rigorous. The article will be rated as not rigorous if any criteria are not met. The criteria are 1) in English, 2) about humans, 3) about topics that are important to the clinical practice of medicine, nursing, rehabilitation, and other health professions, other than descriptive studies of prevalence, 4) analysis of each article consistent with the study question, 5) random allocation of participants to comparison groups, 6) 10 or more patients per group completing primary outcome assessment, 7) primary outcome(s) assessed in 80% or more of those randomized at the defined follow-up point, 8) primary outcome is clinically important or ≥1 secondary outcome is clinically important, and 9) subgroup analyses must be preplanned, with groups analyzed as they were randomized; analyses must test for interaction between 2 or more subgroups.
"""

BATCH_SIZE = 20

## Index Perturbation

In [ ]:
class TokenImportance(BaseModel):
    token_index: int
    importance_value: float


class TokenImportanceResults(BaseModel):
    values: list[TokenImportance]


PROMPT_PROVIDED_INFO = f"""
You will be provided the number of tokens, the logits for both positive and negative classes, and the probability for the positive class. You will NOT be provided the text.
"""

PROMPT_INSTRUCTIONS_VARIABLE = f"""
For a given instance, determine which tokens have the greatest impact on the model's prediction by systematically masking them. You will:

1. Receive an instance with the number of tokens and model outputs without any masking.
2. Generate a list of lists of indexes to mask. Tokens corresponding to the indexes will be replaced by `[MASK]` by the `mask_and_predict` function. Please cover as many combinations of masking as possible. You must generate at least {NUM_LIST_MIN} list(s) and at least {NUM_ITEM} index(es) in each list per iteration.
3. Call `mask_and_predict` with the lists of indexes to mask. The function will return a list of lists in the form of [logit_positive, logit_negative, probability_positive], in which each list corresponds with the model's output given the masked variant.
4. Determine whether another iteration of masking is needed for additional information. If yes, repeat steps 2 and 3, adjust masking based on results, and DO NOT reply with anything else. Ensure to cover all tokens at least once with different masking combinations. If no, proceed to the next step. Please use at least {MIN_I} iterations, but DO NOT exceed {MAX_I} iterations.
5. Calculate the importance of each token in the text, where the importance should be a float. A negative and positive float should indicate that the token increases the chance of a negative and positive classification, respectively.
6. Output a list of importance values, where the order of the importance values should correspond to the order of the tokens. The length of the list MUST be equal to the number of tokens. Please do not reply with anything else.

Generate the masks and call the `mask_and_predict` function in each iteration judiciously. Please cover as many combinations of masking as possible.
"""

PROMPT_INSTRUCTIONS_FIXED = f"""
For a given instance, determine which tokens have the greatest impact on the model's prediction by systematically masking them. You will:

1. Receive an instance with the number of tokens and model outputs without any masking.
2. Define 'importance' for yourself. The importance should be a float. A negative and positive float should indicate that the token increases the chance of a negative and positive classification, respectively.
3. Generate a list of num_token number of lists where each list contains the index of 1 token to mask (e.g., [[0], [1], [2], [3], ...]). Tokens corresponding to the indexes will be replaced by `[MASK]` by the `mask_and_predict` function.
4. Repeat steps 4a and 4b for {FIXED_I} iterations. Once all {FIXED_I} iterations are complete, you will be prompted to proceed to step 5.
4a. Generate a list of lists of one or numerous index(es) to mask, based on the results of previous iterations. Start with completely random number of lists and indexes and adjust the number of lists and indexes in each list based on model outputs. DO NOT repeatedly mask the same combination of tokens.
4b. Call `mask_and_predict` with the lists of indexes to mask. The function will return a list of lists in the form of [logit_positive, logit_negative, probability_positive], in which each list corresponds with the model's output given the masked variant.
5. You will make any adjustments to your initial definition of 'importance'.
6. You will be prompted with indexes of the tokens. Please calculate the importance of each prompted token in the text.
7. For each token, output the index and the importance.
"""

PROMPT_DEVELOPER = f"""
{PROMPT_ROLE} {PROMPT_TASK}
{PROMPT_PROVIDED_INFO}
{PROMPT_INSTRUCTIONS_FIXED}
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "mask_and_predict",
            "description": "This function takes a list of lists of indexes (e.g., [[0, 11, 57], [1, 7, 23, 326], [2, 4, 8, 9, 10, 13, 24, 45, 76, 129, 131, 145, 246, ...], ...]) and returns a list of lists of three floats (logit_positive, logit_negative, probability_positive). For each list of indexes, the token corresponding to the indexes will be replaced with '[MASK]', and the model will be used to predict with the masked tokens and generate the tuple.",
            "strict": True,
            "parameters": {
                "type": "object",
                "required": ["mask_lists"],
                "additionalProperties": False,
                "properties": {
                    "mask_lists": {
                        "type": "array",
                        "items": {
                            "type": "array",
                            "items": {
                                "type": "integer"
                            }
                        }
                    }
                }
            }
        }
    }
]

# Resume support: skip IDs that already have saved outputs.
result_dir = Path("gpt_index_masking") / "results"
message_dir = Path("gpt_index_masking") / "messages"
result_dir.mkdir(parents=True, exist_ok=True)
message_dir.mkdir(parents=True, exist_ok=True)

result_files = [p.name for p in result_dir.glob("*.json")]
prev_ids = [int(r.replace(".json", "")) for r in result_files]
print(prev_ids)

ids = [i for i in df["id"].unique() if i in data["id"] and i not in prev_ids]
print(len(ids))

for id in tqdm(ids):
    print(id)
    df_temp = df[df["id"] == id]
    responses = []

    message_path = message_dir / f"{id}.pkl"
    if message_path.is_file():
        print(f"Message file found for {id}.")
        with message_path.open("rb") as file:
            messages = pickle.load(file)
    else:
        PROMPT_USER = f"""
        Number of tokens: {len(df_temp)}.

        Model output without masking: {mask_and_predict(list(df_temp['sv_token']), [[]])[0][0]}.
        """

        messages = [
            {"role": "developer", "content": PROMPT_DEVELOPER},
            {"role": "user", "content": PROMPT_USER},
            {
                "role": "user",
                "content": "Please define 'importance'. You will use this definition to calculate token importance when prompted later.",
            },
        ]

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            max_tokens=16383,
        )

        print(response.choices[0].message)
        responses.append(response)
        messages.append(response.choices[0].message)

        messages.append(
            {
                "role": "user",
                "content": f"Please generate the initial masking list and call 'mask_and_predict'. The input must be a list of {len(df_temp)} lists of one index each.",
            }
        )

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            tools=tools,
            max_tokens=16383,
        )
        print(response.choices[0].message)

        responses.append(response)
        messages.append(response.choices[0].message)

        tool_call = response.choices[0].message.tool_calls[0]
        arguments = json.loads(tool_call.function.arguments)
        mask_lists = arguments.get("mask_lists")

        masked_probabilities, masked_words, masked_texts = mask_and_predict(
            list(df_temp["sv_token"]), mask_lists
        )

        function_call_result_message = {
            "role": "tool",
            "content": json.dumps({"model_output": masked_probabilities}),
            "tool_call_id": response.choices[0].message.tool_calls[0].id,
        }

        messages.append(function_call_result_message)

        iteration = 1
        while iteration <= FIXED_I:
            messages.append(
                {
                    "role": "user",
                    "content": f"Please proceed with iteration {iteration} of masking. Generate {NUM_LIST_MIN} to {NUM_LIST_MAX} lists. Only generate indexes from 0 to {len(df_temp) - 1}. Add, remove, or adjust masking based on previous maskings and model outputs. Prioritize masking tokens that seemed to be important from previous iterations AND tokens that have rarely been masked in previous iterations. DO NOT repeatedly mask the same combination of tokens. DO NOT reply with anything else other than the function call.",
                }
            )

            response = client.chat.completions.create(
                model=GPT_MODEL,
                temperature=TEMPERATURE,
                presence_penalty=PRESENCE_PENALTY,
                frequency_penalty=FREQUENCY_PENALTY,
                messages=messages,
                tools=tools,
                max_tokens=16383,
            )
            print(response.choices[0].message)

            messages.append(response.choices[0].message)
            responses.append(response)

            if response.choices[0].message.tool_calls is None:
                raise Exception("Function call missing!")

            tool_call = response.choices[0].message.tool_calls[0]
            arguments = json.loads(tool_call.function.arguments)
            mask_lists = arguments.get("mask_lists")

            masked_probabilities, masked_words, masked_texts = mask_and_predict(
                list(df_temp["sv_token"]), mask_lists
            )

            function_call_result_message = {
                "role": "tool",
                "content": json.dumps({"model_output": masked_probabilities}),
                "tool_call_id": response.choices[0].message.tool_calls[0].id,
            }

            messages.append(function_call_result_message)
            iteration += 1

        messages.append(
            {
                "role": "user",
                "content": "Based on the results of all previous masking iterations, adjust the definition of 'importance' for yourself as you see fit.",
            }
        )

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            tools=tools,
            max_tokens=16383,
        )

        messages.append(response.choices[0].message)
        responses.append(response)

        with message_path.open("wb") as file:
            pickle.dump(messages, file)

    # Final scoring is requested in token batches to reduce response size.
    instance_output = {}
    messages.append({})

    for i in range(0, len(df_temp), BATCH_SIZE):
        messages.pop(-1)
        messages.append(
            {
                "role": "user",
                "content": f"Please calculate token importance for tokens from index {i} to {min(i + BATCH_SIZE - 1, len(df_temp) - 1)} based on previous information and your definition of 'importance'.",
            }
        )

        response = client.beta.chat.completions.parse(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            max_tokens=16383,
            response_format=TokenImportanceResults,
        )

        answer = response.choices[0].message.parsed
        for item in answer.values:
            instance_output[item.token_index] = item.importance_value

    with (result_dir / f"{id}.json").open("w") as json_file:
        json.dump(instance_output, json_file, indent=2)

# Index Word Masking

In [ ]:
class TokenImportance(BaseModel):
    token_index: int
    importance_value: float


class Step(BaseModel):
    self_explanation: str


class TokenImportanceResults(BaseModel):
    values: list[TokenImportance]


PROMPT_PROVIDED_INFO = f"""
You will be provided the number of tokens, the input tokens, the logits for both positive and negative classes, the probability for the positive class. You will also be provided the manual appraisal criteria (appraised using the full text).
"""

PROMPT_INSTRUCTIONS_FIXED = f"""
For a given instance, determine which tokens have the greatest impact on the model's prediction by systematically masking them. You will:

1. Receive an instance with the number of tokens, the tokens, and model outputs without any masking.
2. Define 'importance' for yourself. The importance should be a float. A negative and positive float should indicate that the token increases the chance of a negative and positive classification, respectively.
3. Generate a list of num_token number of lists where each list contains the index of 1 token to mask (e.g., [[0], [1], [2], [3], ...]). Tokens corresponding to the indexes will be replaced by `[MASK]` by the `mask_and_predict` function. 
4. Repeat steps 4a and 4b for {FIXED_I} iterations. Once all {FIXED_I} iterations are complete, you will be prompted to proceed to step 5.
4a. Generate a list of lists of one or numerous index(es) to mask, based on which token you think is semantically important and the results of previous masking iterations. DO NOT repeatedly mask the same combination of tokens.
4b. Call `mask_and_predict` with the lists of indexes to mask. The function will return a list of lists in the form of [logit_positive, logit_negative, probability_positive], in which each list corresponds with the model's output given the masked variant.
5. You will make any adjustments to your initial definition of 'importance'.
6. You will be prompted with indexes of the tokens. Please calculate the importance of each prompted token in the text.
7. For each token, output the index and the importance.
"""

PROMPT_DEVELOPER = f"""
{PROMPT_ROLE} {PROMPT_TASK}
{PROMPT_PROVIDED_INFO}
{PROMPT_INSTRUCTIONS_FIXED}
{PROMPT_CRITERIA}
"""

print(PROMPT_DEVELOPER)

tools = [
    {
        "type": "function",
        "function": {
            "name": "mask_and_predict",
            "description": "This function takes a list of lists of indexes (e.g., [[0, 11, 57], [1, 7, 23, 326], [2, 4, 8, 9, 10, 13, 24, 45, 76, 129, 131, 145, 246, ...], ...]) and returns a list of lists of three floats (logit_positive, logit_negative, probability_positive). For each list of indexes, the token corresponding to the indexes will be replaced with '[MASK]', and the model will be used to predict with the masked tokens and generate the tuple.",
            "strict": True,
            "parameters": {
                "type": "object",
                "required": ["mask_lists"],
                "additionalProperties": False,
                "properties": {
                    "mask_lists": {
                        "type": "array",
                        "items": {
                            "type": "array",
                            "items": {
                                "type": "integer"
                            }
                        }
                    }
                }
            }
        }
    }
]

# Resume support: skip IDs that already have saved outputs.
result_dir = Path("gpt_index_word_masking") / "results"
message_dir = Path("gpt_index_word_masking") / "messages"
message_all_dir = Path("gpt_index_word_masking") / "messages_all"
result_dir.mkdir(parents=True, exist_ok=True)
message_dir.mkdir(parents=True, exist_ok=True)
message_all_dir.mkdir(parents=True, exist_ok=True)

result_files = [p.name for p in result_dir.glob("*.json")]
prev_ids = [int(r.replace(".json", "")) for r in result_files]

ids = [i for i in df["id"].unique() if i in data["id"] and i not in prev_ids]
messages_all = []

for id in tqdm(ids):
    print(id)
    df_temp = df[df["id"] == id]
    responses = []

    message_path = message_dir / f"{id}.pkl"
    if message_path.is_file():
        print(f"Message file found for {id}.")
        with message_path.open("rb") as file:
            messages = pickle.load(file)
    else:
        PROMPT_USER = f"""
        Number of tokens: {len(df_temp)}.
        Input tokens: {list(df_temp['ig_token'])}
        Model output without masking: {mask_and_predict(list(df_temp['sv_token']), [[]])[0][0]}.
        """
        print(PROMPT_USER)

        messages = [
            {"role": "developer", "content": PROMPT_DEVELOPER},
            {"role": "user", "content": PROMPT_USER},
            {
                "role": "user",
                "content": "Please define 'importance'. You will use this definition to calculate token importance when prompted later.",
            },
        ]

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            max_tokens=16383,
        )

        print(response.choices[0].message)
        responses.append(response)
        messages.append(response.choices[0].message)

        messages.append(
            {
                "role": "user",
                "content": f"Please generate the initial masking list and call 'mask_and_predict'. The input must be a list of {len(df_temp)} lists of one index each.",
            }
        )

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            tools=tools,
            max_tokens=16383,
        )
        print(response.choices[0].message)

        responses.append(response)
        messages.append(response.choices[0].message)

        tool_call = response.choices[0].message.tool_calls[0]
        arguments = json.loads(tool_call.function.arguments)
        mask_lists = arguments.get("mask_lists")

        masked_probabilities, masked_words, masked_texts = mask_and_predict(
            list(df_temp["sv_token"]), mask_lists
        )

        function_call_result_message = {
            "role": "tool",
            "content": json.dumps({"model_output": masked_probabilities}),
            "tool_call_id": response.choices[0].message.tool_calls[0].id,
        }
        messages.append(function_call_result_message)

        iteration = 1
        while iteration <= FIXED_I:
            messages.append(
                {
                    "role": "user",
                    "content": f"Please proceed with iteration {iteration} of masking. Generate {NUM_LIST_MIN} to {NUM_LIST_MAX} lists. Only generate indexes from 0 to {len(df_temp) - 1}. Add, remove, or adjust masking based on previous maskings and model outputs. Prioritize masking tokens that seemed to be important from previous iterations AND tokens that have rarely been masked in previous iterations. DO NOT repeatedly mask the same combination of tokens. DO NOT reply with anything else other than the function call.",
                }
            )

            response = client.chat.completions.create(
                model=GPT_MODEL,
                temperature=TEMPERATURE,
                presence_penalty=PRESENCE_PENALTY,
                frequency_penalty=FREQUENCY_PENALTY,
                messages=messages,
                tools=tools,
                max_tokens=16383,
            )
            print(response.choices[0].message)

            messages.append(response.choices[0].message)
            responses.append(response)

            if response.choices[0].message.tool_calls is None:
                raise Exception("Function call missing!")

            tool_call = response.choices[0].message.tool_calls[0]
            arguments = json.loads(tool_call.function.arguments)
            mask_lists = arguments.get("mask_lists")

            masked_probabilities, masked_words, masked_texts = mask_and_predict(
                list(df_temp["sv_token"]), mask_lists
            )

            function_call_result_message = {
                "role": "tool",
                "content": json.dumps({"model_output": masked_probabilities}),
                "tool_call_id": response.choices[0].message.tool_calls[0].id,
            }
            messages.append(function_call_result_message)

            iteration += 1

        messages.append(
            {
                "role": "user",
                "content": "Based on the results of all previous masking iterations, adjust the definition of 'importance' for yourself as you see fit.",
            }
        )

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            tools=tools,
            max_tokens=16383,
        )
        print(response.choices[0].message)

        messages.append(response.choices[0].message)
        responses.append(response)

        with message_path.open("wb") as file:
            pickle.dump(messages, file)

    messages_all.extend(messages)

    # Final scoring is requested in token batches to reduce response size.
    instance_output = {}
    messages.append({})

    for i in range(0, len(df_temp), BATCH_SIZE):
        messages.pop(-1)
        messages.append(
            {
                "role": "user",
                "content": f"Please calculate token importance for tokens from index {i} to {min(i + BATCH_SIZE - 1, len(df_temp) - 1)} based on previous information and your definition of 'importance'.",
            }
        )
        print(
            f"Please calculate token importance for tokens from index {i} to {min(i + BATCH_SIZE - 1, len(df_temp) - 1)} based on previous information and your definition of 'importance'."
        )
        messages_all.append(messages[-1])

        try:
            response = client.beta.chat.completions.parse(
                model=GPT_MODEL,
                temperature=TEMPERATURE,
                presence_penalty=PRESENCE_PENALTY,
                frequency_penalty=FREQUENCY_PENALTY,
                messages=messages,
                max_tokens=16383,
                response_format=TokenImportanceResults,
            )
            print(response.choices[0].message.parsed)
            answer = response.choices[0].message.parsed
            for item in answer.values:
                instance_output[item.token_index] = item.importance_value
        except Exception as e:
            print(e)

    with (result_dir / f"{id}.json").open("w") as json_file:
        json.dump(instance_output, json_file, indent=2)

    with (message_all_dir / f"{id}.pkl").open("wb") as file:
        pickle.dump(messages_all, file)

In [ ]:
print(response.choices[0].message.parsed)

# Archive

In [ ]:
import gpt_response_formats

class TokenImportance (BaseModel):
    token_index: int
    importance_value: float


class Step (BaseModel):
    self_explanation: str


class TokenImportanceResults (BaseModel):
    values: list[TokenImportance]


PROMPT_PROVIDED_INFO = f"""
You will be provided the number of tokens, the logits for both positive and negative classes, and the probability for the positive class. You will NOT be provided the text.
"""
PROMPT_INSTRUCTIONS_VARIABLE = f"""
For a given instance, determine which tokens have the greatest impact on the model's prediction by systematically masking them. You will:

1. Receive an instance with the number of tokens and model outputs without any masking.
2. Generate a list of lists of indexes to mask. Tokens corresponding to the indexes will be replaced by `[MASK]` by the `mask_and_predict` function. Please cover as many combinations of masking as possible. You must generate at least {NUM_LIST_MIN} list(s) and at least {NUM_ITEM} index(es) in each list per iteration.
3. Call `mask_and_predict` with the lists of indexes to mask. The function will return a list of lists in the form of [logit_positive, logit_negative, probability_positive], in which each list corresponds with the model's output given the masked variant.
4. Determine whether another iteration of masking is needed for additional information. If yes, repeat steps 2 and 3, adjust masking based on results, and DO NOT reply with anything else. Ensure to cover all tokens at least once with different masking combinations. If no, proceed to the next step. Please use at least {MIN_I} iterations, but DO NOT exceed {MAX_I} iterations.
5. Calculate the importance of each token in the text, where the importance should be a float. A negative and positive float should indicate that the token increases the chance of a negative and positive classification, respectively.
6. Output a list of importance values, where the order of the importance values should correspond to the order of the tokens. The length of the list MUST be equal to the number of tokens. Please do not reply with anything else.

Generate the masks and call the `mask_and_predict` function in each iteration judiciously. Please cover as many combinations of masking as possible.
"""

PROMPT_INSTRUCTIONS_FIXED = f"""
For a given instance, determine which tokens have the greatest impact on the model's prediction by systematically masking them. You will:

1. Receive an instance with the number of tokens and model outputs without any masking.
2. Define 'importance' for yourself. The importance should be a float. A negative and positive float should indicate that the token increases the chance of a negative and positive classification, respectively.
3. Generate a list of num_token number of lists where each list contains the index of 1 token to mask (e.g., [[0], [1], [2], [3], ...]). Tokens corresponding to the indexes will be replaced by `[MASK]` by the `mask_and_predict` function.
4. Repeat steps 4a and 4b for {FIXED_I} iterations. Once all {FIXED_I} iterations are complete, you will be prompted to proceed to step 5.
4a. Generate a list of lists of one or numerous index(es) to mask, based on the results of previous iterations. Start with completely random number of lists and indexes and adjust the number of lists and indexes in each list based on model outputs.
4b. Call `mask_and_predict` with the lists of indexes to mask. The function will return a list of lists in the form of [logit_positive, logit_negative, probability_positive], in which each list corresponds with the model's output given the masked variant.
5. You will make any adjustments to your initial definition of 'importance'.
6. You will be prompted with indexes of the tokens. Please calculate the importance of each prompted token in the text.
7. For each token, output the index and the importance. If the importance of a token cannot not be assessed, output 0 as the importance.
"""

PROMPT_DEVELOPER = f"""
{PROMPT_ROLE} {PROMPT_TASK}
{PROMPT_PROVIDED_INFO}
{PROMPT_INSTRUCTIONS_FIXED}
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "mask_and_predict",
            "description": "This function takes a list of lists of indexes (e.g., [[0, 11, 57], [1, 7, 23, 326], [2, 4, 8, 9, 10, 13, 24, 45, 76, 129, 131, 145, 246, ...], ...]) and returns a list of lists of three floats (logit_positive, logit_negative, probability_positive). For each list of indexes, the token corresponding to the indexes will be replaced with '[MASK]', and the model will be used to predict with the masked tokens and generate the tuple.",
            "strict": True,
            "parameters": {
                "type": "object",
                "required": ["mask_lists"],
                "additionalProperties": False,
                "properties": {
                    "mask_lists": {
                        "type": "array",
                        "items": {
                            "type": "array",
                            "items": {
                                "type": "integer"
                            }
                        }
                    },
                }
            }
        }
    }
]

id = 35112672

df_temp = df[df['id'] == id]
responses = []
# print(text)

PROMPT_USER = f"""
Number of tokens: {len(df_temp)}.

Model output without masking: {mask_and_predict(list(df_temp['sv_token']), [[]])[0][0]}.
"""
# print(PROMPT_USER)

messages = [
    {
        "role": "developer",
        "content": PROMPT_DEVELOPER
    },
    {
        "role": "user",
        "content": PROMPT_USER
    },
    {
        "role": "user",
        "content": f"Please define 'importance'. You will use this definition to calculate token importance when prompted later."
    }
]

response = client.chat.completions.create(
    model=GPT_MODEL,
    temperature=TEMPERATURE,
    presence_penalty=PRESENCE_PENALTY,
    frequency_penalty=FREQUENCY_PENALTY,
    messages=messages,
    max_tokens=16383
)

print(response.choices[0].message)
responses.append(response)
messages.append(response.choices[0].message)

messages.append({
    "role": "user",
    "content": f"Please generate the initial masking list and call 'mask_and_predict'. The input must be a list of {len(df_temp)} lists of one index each."
})

response = client.chat.completions.create(
    model=GPT_MODEL,
    temperature=TEMPERATURE,
    presence_penalty=PRESENCE_PENALTY,
    frequency_penalty=FREQUENCY_PENALTY,
    messages=messages,
    tools=tools,
    max_tokens=16383
)
print(response.choices[0].message)

responses.append(response)
messages.append(response.choices[0].message)

tool_call = response.choices[0].message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)

mask_lists = arguments.get('mask_lists')

masked_probabilities, masked_words, masked_texts = mask_and_predict(list(df_temp['sv_token']), mask_lists)

function_call_result_message = {
    "role": "tool",
    "content": json.dumps({
        # "mask_lists": mask_lists,
        "model_output": masked_probabilities,
        # "masked_words": masked_words
    }),
    "tool_call_id": response.choices[0].message.tool_calls[0].id
}
# print(function_call_result_message)

messages.append(function_call_result_message)

iteration = 1
while iteration <= FIXED_I:
    # print(f"iteration: {iteration}")

    messages.append({
        "role": "user",
        "content": f"Please proceed with iteration {iteration} of masking. Generate {NUM_LIST_MIN} to {NUM_LIST_MAX} lists. Only generate indexes from 0 to {len(df_temp) - 1}. Adjust the number of lists and indexes in each list based on previous maskings and model outputs. Prioritize masking tokens that seemed to be important from previous iterations AND tokens that have rarely been masked in previous iterations. DO NOT repeatedly mask the same list of tokens. DO NOT reply with anything else other than the function call."
    })

    response = client.chat.completions.create(
        model=GPT_MODEL,
        temperature=TEMPERATURE,
        presence_penalty=PRESENCE_PENALTY,
        frequency_penalty=FREQUENCY_PENALTY,
        messages=messages,
        tools=tools,
        max_tokens=16383
    )
    print(response.choices[0].message)

    messages.append(response.choices[0].message)
    responses.append(response)

    if response.choices[0].message.tool_calls == None:
        raise Exception("Function call missing!")

    tool_call = response.choices[0].message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)

    mask_lists = arguments.get('mask_lists')

    masked_probabilities, masked_words, masked_texts = mask_and_predict(list(df_temp['sv_token']), mask_lists)
    # print(masked_probabilities)
    # print(masked_words)
    # print(masked_texts)

    function_call_result_message = {
        "role": "tool",
        "content": json.dumps({
            # "mask_lists": mask_lists,
            "model_output": masked_probabilities,
            # "masked_words": masked_words
        }),
        "tool_call_id": response.choices[0].message.tool_calls[0].id
    }
    # print(function_call_result_message)

    messages.append(function_call_result_message)

    iteration += 1

# print(response.choices[0].message)

messages.append({
    "role": "user",
    "content": f"Based on the results of all previous masking iterations, adjust the definition of 'importance' for yourself as you see fit."
})

response = client.chat.completions.create(
    model=GPT_MODEL,
    temperature=TEMPERATURE,
    presence_penalty=PRESENCE_PENALTY,
    frequency_penalty=FREQUENCY_PENALTY,
    messages=messages,
    tools=tools,
    max_tokens=16383
)
# print(response.choices[0].message)

messages.append(response.choices[0].message)
responses.append(response)

for r in responses:
    print(r)

# with (Path("responses") / f"{id}.pkl").open("wb") as file:
#     pickle.dump(responses, file)

# with (Path("messages") / f"{id}.pkl").open("wb") as file:
#     pickle.dump(messages, file)

instance_output = {}
messages.append({})

for i in range(0, len(df_temp), BATCH_SIZE):
    messages.pop(-1)
    messages.append({
        "role": "user",
        "content": f"Please calculate token importance for tokens from index {i} to {min(i + BATCH_SIZE - 1, len(df_temp) - 1)} based on previous information and your definition of 'importance'."
    })
    print(f"Please calculate token importance for tokens from index {i} to {min(i + BATCH_SIZE - 1, len(df_temp) - 1)} based on previous information and your definition of 'importance'.")
    print(messages)

    response = client.beta.chat.completions.parse(
        model=GPT_MODEL,
        temperature=TEMPERATURE,
        presence_penalty=PRESENCE_PENALTY,
        frequency_penalty=FREQUENCY_PENALTY,
        messages=messages,
        max_tokens=16383,
        response_format=TokenImportanceResults,
    )

    answer = response.choices[0].message.parsed
    # print(answer)
    for i in answer.values:
        instance_output[i.token_index] = i.importance_value
    
    for o in instance_output:
        print(o, instance_output[o])

    # with (Path("responses") / f"{id}.json").open("w") as json_file:
    #     json.dump(instance_output, json_file, indent=2)

In [ ]:
import pickle
from pathlib import Path

with (Path("gpt_index_word_masking") / "messages" / "10632803.pkl").open("rb") as file:
    messages = pickle.load(file)

for m in messages:
    print(m)

In [ ]:
df = pd.read_csv("feature_attributions.csv")
df.head()

In [ ]:
DEVELOPER_PERTURBATION_WORD = f"""

You are a machine learning model explainer. You are tasked with explaining an encoder-only transformer model's predictions by perturbing its input text, similar to SHAP. The model was trained on the title and abstract of medical literature, each labelled rigorous or not rigorous based on the criteria provided below. You will be provided the text, the total number of words, and the probability for the rigorous class for the text without any masking.

For a given input text, determine which words or phrases have the greatest impact on the model's prediction by systematically masking them. You will:
1. Receive the input text.
2. Generate one or more lists of words, phrases (series of words, n-grams) and other combinations to mask. Matching words in the text will automatically be replaced by `[MASK]` by the `mask_and_predict` function. You must include at least {NUM_LIST} list(s) of words and at least {NUM_ITEM} word(s) in each list.
3. Call `mask_and_predict` with the masked texts `mask_and_predict(lists)` to evaluate the probabilities of the rigorous class for the given masked variants.
4. Determine whether another iteration of masking is needed for additional information. If yes, adjust masking based on previous results by thinking critically of how previous masking influenced predicted probability, repeat steps 2 and 3 and DO NOT reply with anything else. If no, do not reply with your rationale and proceed to the next step. Please use at least {MIN_I} iterations, but DO NOT exceed {MAX_I} iterations.
5. Calculate the influence on the rigorous probability of every word in the text, where influence is defined as the probability with the word masked minus the probability with the word unmasked. Note that some words have positive influence (masking would decrease the rigorous probability) and others have negative influence (masking would increase the rigorous probability).
6. Output a list of **EVERY** word from the original text and their average influences. Rank from highest absolute influence to lowest absolute influence and only output one word per line, in the form of '1. <word>: <influence>; \n 2. <word>: <influence>; \n ...' DO NOT output anything else.

Generate the masks and call the `mask_and_predict` function in each iteration judiciously.

Manual appraisal criteria:
All of these criteria must be met to be rated as being rigorous. The article will be rated as not rigorous if any criteria are not met. The criteria are 1) in English, 2) about humans, 3) about topics that are important to the clinical practice of medicine, nursing, rehabilitation, and other health professions, other than descriptive studies of prevalence, 4) analysis of each article consistent with the study question, 5) random allocation of participants to comparison groups, 6) 10 or more patients per group completing primary outcome assessment, 7) primary outcome(s) assessed in 80% or more of those randomized at the defined follow-up point, 8) primary outcome is clinically important or ≥1 secondary outcome is clinically important, and 9) subgroup analyses must be preplanned, with groups analyzed as they were randomized; analyses must test for interaction between 2 or more subgroups.
"""

tools = [
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "mask_and_predict",
    #         "description": "This function takes a list of lists of indexes (e.g., [[0, 11, 57], [1, 2, 23, 123, 468], [2, 4, 8, 9, 10, 13, 24, 45, 76,129, 131, 145, 246, ...], ...]) and returns a list of floats. For each list of indexes, the words in the text corresponding to the indexes will be replaced with '[MASK]', and the deep learning model will be used to predict the probabilities of positive class based on the text for every list of masked words.",
    #         "strict": False,
    #         "parameters": {
    #             "type": "object",
    #             "lists": {
    #                 "type": "array",
    #                 "items": {
    #                     "type": "array",
    #                     "items": {
    #                         "type": "integer"
    #                     }
    #                 }
    #             },
    #             "required": [
    #                 "lists"
    #             ],
    #             "additionalProperties": False,
    #             "properties": {}
    #         }
    #     }
    # }
    {
        "type": "function",
        "function": {
            "name": "mask_and_predict",
            "description": "This function takes a list of lists of words (e.g., [['randomized'], ['norepinephrine', 'random', 'retrospective', ...], ...]) and returns a list of floats. For each list of words, all words in the text will be compared. If a match is found, the word in the text is replaced with '[MASK]', and the deep learning model will be used to predict the probabilities of positive class based on the text for every list of masked words.",
            "strict": False,
            "parameters": {
                "type": "object",
                "lists": {
                    "type": "array",
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "lists"
                ],
                "additionalProperties": False,
                "properties": {}
            }
        }
    }
]

df = pd.read_csv("feature_attributions.csv")
df = df.head(2)

for _, row in df.iterrows():
    text = row['text'].split()[:512]
    text = " ".join(text)
    id = row['id']
    print(f"{id}, {count_words(text)}")

    USER_TEST = f"""

    Text: {text}

    Number of words: {count_words(text)}

    Probability of positive class: {predict.mask_and_predict(text, [[]])}.
    """

    messages = [
        {
            "role": "developer",
                    "content": DEVELOPER_PERTURBATION_WORD
        },
        {
            "role": "user",
                    "content": USER_TEST
        }
    ]

    responses = []

    response = client.chat.completions.create(
        model=GPT_MODEL,
        temperature=TEMPERATURE,
        presence_penalty=PRESENCE_PENALTY,
        frequency_penalty=FREQUENCY_PENALTY,
        messages=messages,
        tools=tools,
        max_tokens=16383

    )

    responses.append(response)

    iteration = MAX_I

    while response.choices[0].message.tool_calls != None and iteration > 0:
        print(f"iteration: {iteration}")
        print(response.choices[0].message)
        tool_call = response.choices[0].message.tool_calls[0]
        arguments = json.loads(tool_call.function.arguments)

        lists = arguments.get('lists')

        masked_probabilities, masked_words, masked_texts = predict.mask_and_predict(text, lists)

        function_call_result_message = {
            "role": "tool",
            "content": json.dumps({
                "lists": lists,
                "masked_probabilities": masked_probabilities,
                # "masked_words": masked_words
            }),
            "tool_call_id": response.choices[0].message.tool_calls[0].id
        }

        messages.append(response.choices[0].message)
        messages.append(function_call_result_message)

        print(function_call_result_message)

        response = client.chat.completions.create(
            model=GPT_MODEL,
            temperature=TEMPERATURE,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            messages=messages,
            tools=tools,
            max_tokens=16383
        )

        responses.append(response)
        iteration -= 1

    print(response.choices[0].message)

    with (Path("responses") / f"{id}.pkl").open("wb") as file:
        pickle.dump(responses, file)